# UL2ME007 – Algorithmics and Scientific Programming｜UL2ME007 – 算法与科学编程

# Course 5: Boolean Masks｜第 5 课：布尔掩码

Combined with vectorisation, a **Boolean mask** can replace many combinations of loops and `if` statements with a compact array expression. The syntax is close to mathematical conditions, but correct use requires a clear understanding of Boolean values, array shapes, and logical operators.  
布尔掩码与向量化结合后，可以用紧凑的数组表达式替代许多循环和 `if` 判断。它的写法接近数学条件，但要正确使用，必须理解布尔值、数组形状以及逻辑运算符的工作方式。

We begin with a cyclist-speed example, then unpack every step and finally compare scalar short-circuit logic with element-wise NumPy logic.  
本课先从自行车速度示例出发，再逐步拆解掩码的内部过程，最后比较标量的短路逻辑与 NumPy 数组的逐元素逻辑。

### Course map｜课程导航

1. [First example: selecting subsets of vectors](#section1)｜从向量中筛选子集  
2. [How Boolean masks work](#section2)｜布尔掩码的工作原理  
3. [Combining logical conditions](#section3)｜组合多个逻辑条件  
4. [Integrated practice](#integrated-practice)｜综合练习


## Learning guide｜学习指南

### Boolean masks and conditional array selection｜布尔掩码与数组条件筛选

By the end of the lesson, you should be able to build a Boolean array from a condition, use it to select or modify matching values, count matches, and combine several NumPy conditions safely with `&`, `|`, and `~`.  
完成本课后，你应能够根据条件构造布尔数组，用它筛选或修改满足条件的元素，统计满足条件的数量，并能使用 `&`、`|` 与 `~` 安全地组合多个 NumPy 条件。

> Before running a cell, predict the mask itself—not only the selected values. Check that the mask has the same shape as the array it indexes.  
> 运行代码前，不仅要预测筛选结果，还要先预测掩码本身，并检查掩码与被索引数组的形状是否一致。


<a id="section1"></a>
## 1 — First example: selecting subsets of vectors｜1 — 第一个示例：从向量中筛选子集

We model the speed of a mountain cyclist during a six-minute journey. Measurements are sampled every five seconds, giving 73 time points including both endpoints. The function is artificial, but it lets us practise selecting meaningful portions of a signal.  
我们用一个人为构造的函数模拟山地自行车手在 6 分钟内的速度。数据每 5 秒采样一次，包含首尾时刻，因此共有 73 个时间点。虽然函数本身只是示例，但很适合练习从信号中筛选有意义的区段。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Measurements every 5 seconds from 0 to 6 minutes, including both endpoints.
t = np.linspace(0, 6, 73)
v = 5 * (np.sin(t) + 1.8 * np.sin(4 * t) + 3)  # example speed in km/h
sample_period_s = (t[1] - t[0]) * 60

def plot_speed(time_min, speed_kmh):
    """Plot cyclist speed (km/h) against time (minutes)."""
    plt.plot(time_min, speed_kmh, ".-", label="complete journey")
    plt.grid(alpha=0.3)
    plt.xlim(0, 6)
    plt.ylabel("Speed (km/h)", fontsize=14)
    plt.xlabel("Time (min)", fontsize=14)

plt.figure(1, figsize=(8, 4))
plot_speed(t, v)
plt.title("Cyclist speed during a six-minute journey")
plt.tight_layout()


Suppose we want the times and speeds during the first minute and then highlight this segment on the complete curve. The condition `t <= 1` is evaluated at every time value and becomes a Boolean mask. We first use it to select the corresponding times.  
假设我们要提取最初 1 分钟内的时间和速度，并在完整曲线上突出显示这一段。条件 `t <= 1` 会对每个时间值分别判断，从而形成布尔掩码；下面先用它筛选相应的时间。


In [ ]:
start_mask = t <= 1
t_start = t[start_mask]
print("Mask for the first minute:", start_mask)
print("Selected times (min):", t_start)


Inside `t[...]`, we can supply a Boolean array instead of integer indices or a slice. Read `t[t <= 1]` as: “select the elements of `t` wherever the condition `t <= 1` is true.”  
在 `t[...]` 中，除了整数索引和切片，也可以放入布尔数组。`t[t <= 1]` 可以读作：“保留 `t` 中满足 `t <= 1` 的位置”。

Because `t` and `v` have the same shape, the mask built from `t` can also select the speed values measured at those same positions. This alignment between arrays is essential.  
由于 `t` 与 `v` 形状相同，根据 `t` 构造的掩码也能筛选相同位置上的速度值。数组之间的位置对应关系是这一操作成立的关键。


In [ ]:
v_start = v[start_mask]
print("Selected speeds (km/h):", v_start)
print("Shapes:", t_start.shape, v_start.shape)


A condition on one vector can therefore select aligned values from another vector. To isolate the interval $1\leq t\leq3$, combine two element-wise conditions. Each comparison must be enclosed in parentheses before joining them with `&`.  
因此，一个向量上的条件可以筛选另一个向量中位置对应的值。若要提取 $1\leq t\leq3$ 的区间，需要组合两个逐元素条件；每个比较都应放在括号中，再使用 `&` 连接。


In [ ]:
interval_mask = (t >= 1) & (t <= 3)
t_cut = t[interval_mask]
v_cut = v[interval_mask]

print("Times from 1 to 3 min:", t_cut)
print("Corresponding speeds:", v_cut)


We now overlay the selected intervals on the original curve. The masks determine the data; Matplotlib only controls how the selected data are displayed.  
下面把筛选出的区段叠加到原始曲线上。掩码负责决定“选择哪些数据”，Matplotlib 则负责决定“如何显示这些数据”。


In [ ]:
plt.figure(2, figsize=(8, 4))
plot_speed(t, v)
plt.plot(t_start, v_start, color="black", lw=4, label=r"$t \leq 1$ min")
plt.plot(t_cut, v_cut, color="red", lw=4, label=r"$1 \leq t \leq 3$ min")
plt.title("Time-based selection with Boolean masks")
plt.legend(fontsize=11)
plt.tight_layout()


Next, instead of selecting by time, we select by speed: values at or above 20 km/h are shown with green markers, and values below 10 km/h with gold markers. Using markers without connecting lines prevents separate low- or high-speed periods from being joined misleadingly.  
接下来不再按时间筛选，而是按速度筛选：不低于 20 km/h 的点使用绿色标记，低于 10 km/h 的点使用金色标记。只画标记而不连接线段，可以避免把彼此分离的高速或低速时段错误地连在一起。


In [ ]:
fast_limit_kmh = 20
slow_limit_kmh = 10
fast_mask = v >= fast_limit_kmh
slow_mask = v < slow_limit_kmh

plt.figure(3, figsize=(8, 4))
plot_speed(t, v)
plt.plot(t[fast_mask], v[fast_mask], linestyle="none", marker="o",
         color="green", label="speed ≥ 20 km/h")
plt.plot(t[slow_mask], v[slow_mask], linestyle="none", marker="o",
         color="gold", markeredgecolor="black", label="speed < 10 km/h")
plt.title("Speed-based selection with Boolean masks")
plt.legend(fontsize=11)
plt.tight_layout()


We can also estimate how long the cyclist was at or above 20 km/h. `np.count_nonzero(mask)` counts the `True` samples, and multiplying by the sampling period converts that count to seconds. This is a discrete estimate: it treats each selected sample as representing one sampling interval.  
还可以估算车手速度不低于 20 km/h 的持续时间。`np.count_nonzero(mask)` 统计 `True` 的采样点数量，再乘以采样周期即可换算成秒。需要注意，这是离散估计，相当于让每个满足条件的采样点代表一个采样间隔。


In [ ]:
fast_sample_count = np.count_nonzero(fast_mask)
estimated_fast_duration_s = fast_sample_count * sample_period_s

print("Sampling period:", sample_period_s, "seconds")
print("Number of samples at or above 20 km/h:", fast_sample_count)
print("Estimated duration:", estimated_fast_duration_s, "seconds")


The resulting expressions are compact because NumPy evaluates the condition over the whole array. A loop-based solution is possible, but it would separate the condition, selection, and counting across more lines of code. Compact code is useful only when its logic remains readable, so naming important masks is often better than repeating a long condition.  
这些表达式之所以紧凑，是因为 NumPy 会在整个数组上计算条件。使用循环也能得到相同结果，但条件判断、筛选和计数会分散在更多代码中。紧凑并不等于难读：较长或反复使用的条件最好保存为有意义的掩码变量。


<a id="section2"></a>
## 2 — How Boolean masks work｜2 — 布尔掩码的工作原理


A Boolean value is either `True` or `False`. A scalar comparison such as `3 > 4` produces one Boolean value. An array comparison applies the same comparison element by element and produces a Boolean array.  
布尔值只有 `True` 与 `False` 两种。`3 > 4` 这样的标量比较只产生一个布尔值；数组比较则对每个元素分别判断，最终得到一个布尔数组。


In [ ]:
condition = 3 > 4
print("3 > 4 gives:", condition)
print("Type:", type(condition))


The same idea extends directly to a NumPy vector, matrix, or higher-dimensional array. The output shape matches the input shape because every element receives its own result.  
同样的思想可以直接用于 NumPy 向量、矩阵或更高维数组。由于每个元素都会得到一个判断结果，输出布尔数组的形状与输入数组一致。


In [ ]:
x = np.array([5, 1, 4, 6, 2])
mask = x >= 4

print("x =", x)
print("x >= 4 gives", mask)
print("Mask dtype:", mask.dtype, "shape:", mask.shape)


The array `mask` records whether `x >= 4` is true at every position. Applying `x[mask]` returns the values at the `True` positions. The selected result is one-dimensional and its length equals the number of `True` values.  
数组 `mask` 记录 `x >= 4` 在每个位置是否成立。使用 `x[mask]` 会返回所有 `True` 位置上的数值；筛选结果是一维数组，其长度等于 `True` 的数量。


In [ ]:
selected_x = x[mask]
print("Selected values:", selected_x)


For `x = [5, 1, 4, 6, 2]`, the mask `x >= 4` is `[True, False, True, True, False]`, so the selected result is `[5, 4, 6]`.

| Position｜位置 | 0 | 1 | 2 | 3 | 4 |
|---|---:|---:|---:|---:|---:|
| `x` | 5 | 1 | 4 | 6 | 2 |
| `x >= 4` | True | False | True | True | False |
| Selected｜保留 | 5 | — | 4 | 6 | — |

Think of the mask as a stencil: `True` positions are openings through which values remain visible, while `False` positions are covered. This table replaces the missing illustration referenced by the original handout without losing its explanation.  
可以把掩码想象成一张模板：`True` 对应开孔，数据可以透出；`False` 对应遮挡。上表替代了原讲义中当前项目缺失的示意图片，同时保留了原有解释。


A mask can also modify matching elements in place. The assignment `x[x >= 4] = 9` changes every selected value while leaving the other positions unchanged.  
掩码也可以原地修改满足条件的元素。赋值语句 `x[x >= 4] = 9` 会改变所有被选中的数值，同时保留其余位置不变。


In [ ]:
x_modified = x.copy()
x_modified[x_modified >= 4] = 9

print("Original x:", x)
print("After x_modified[x_modified >= 4] = 9:", x_modified)


The vector `x` is now modified. This is analogous to slice assignment such as `x[1:4] = 9`; the difference is that a Boolean condition chooses the positions instead of a continuous index range.  
此时向量 `x` 已被修改。这与 `x[1:4] = 9` 这样的切片赋值类似，只是选择位置的方式从连续索引区间变成了布尔条件。

To count matches, use `np.count_nonzero(mask)` or `mask.sum()`. In arithmetic contexts, NumPy treats `True` as 1 and `False` as 0, so summing a Boolean array counts its `True` values. `np.count_nonzero` often states the intention more explicitly.  
统计满足条件的数量时，可以使用 `np.count_nonzero(mask)` 或 `mask.sum()`。在数值运算中，NumPy 把 `True` 视为 1、`False` 视为 0，因此布尔数组求和就等于统计 `True`；而 `np.count_nonzero` 往往能更直接地表达意图。


In [ ]:
example_mask = np.array([True, False, True, True, False])
true_count_sum = example_mask.sum()
true_count_explicit = np.count_nonzero(example_mask)

print("Boolean mask:", example_mask)
print("Count using sum:", true_count_sum)
print("Count using count_nonzero:", true_count_explicit)


<a id="section3"></a>
## 3 — Combining logical conditions｜3 — 组合多个逻辑条件

Python provides two related but different families of logical operators:

- `and`, `or`, and `not` operate on single truth values and use **short-circuit evaluation**;
- `&`, `|`, and `~` operate element by element on NumPy Boolean arrays.

Python 中有两组相关但用途不同的逻辑运算符：`and`、`or`、`not` 用于单个真假值，并采用**短路求值**；`&`、`|`、`~` 则用于 NumPy 布尔数组，在每个位置分别计算。

For the scalar expression `(3 > 4) and (7 < 8)`, the first comparison is false, so Python already knows the whole expression is false and does not need the second result. This is short-circuit evaluation. A NumPy mask, however, needs a result at every array position, so use expressions such as `(t >= 1) & (t <= 3)`.  
对于标量表达式 `(3 > 4) and (7 < 8)`，第一个比较已经为假，因此 Python 无需依赖第二个结果就能确定整体为假，这就是短路求值。NumPy 掩码必须在数组的每个位置产生结果，因此应写成 `(t >= 1) & (t <= 3)`。

Parentheses are essential because comparisons and `&` / `|` have different precedence rules. Useful array patterns include:

- **and｜且：** `(a > lower) & (a < upper)`
- **or｜或：** `(a < lower) | (a > upper)`
- **not｜非：** `~(a > limit)`

一个实用规则是：控制流程中的单个条件通常使用 `and`、`or`、`not`；NumPy 数组条件通常使用带括号的 `&`、`|`、`~`。不要对数组直接使用 `and` 或 `or`，否则 Python 无法把整组真假值压缩成唯一的真或假。


In [ ]:
scalar_result = (3 > 4) and (7 < 8)
inside_mask = (t >= 1) & (t <= 3)
outside_mask = (t < 1) | (t > 3)
not_fast_mask = ~(v >= fast_limit_kmh)

print("Scalar short-circuit result:", scalar_result)
print("Samples inside [1, 3] min:", np.count_nonzero(inside_mask))
print("All positions are either inside or outside:", np.all(inside_mask | outside_mask))
print("Not-fast mask is the complement of fast mask:", np.array_equal(not_fast_mask, ~fast_mask))


<a id="integrated-practice"></a>
## 4 — Integrated practice: find a steady riding interval｜4 — 综合练习：寻找稳定骑行区间

Combine time and speed conditions to find measurements between 2 and 5 minutes where the cyclist's speed is between 10 and 20 km/h. This brings together aligned arrays, compound masks, selection, counting, percentages, and plotting.  
组合时间与速度条件，找出 2 至 5 分钟之间、车速处于 10 至 20 km/h 的测量点。这个任务把数组对齐、组合掩码、筛选、计数、百分比和绘图串联到同一流程中。

Before running the cell, predict whether the selected points will form one continuous segment. Why might a line plot be misleading here?  
运行前先判断这些点是否会形成一个连续区段，并思考为什么用线把它们直接连接起来可能产生误导。


In [ ]:
steady_mask = (t >= 2) & (t <= 5) & (v >= 10) & (v <= 20)
steady_t = t[steady_mask]
steady_v = v[steady_mask]
steady_percentage = 100 * np.count_nonzero(steady_mask) / t.size

print("Selected sample count:", np.count_nonzero(steady_mask))
print(f"Percentage of all measurements: {steady_percentage:.1f}%")

plt.figure(4, figsize=(8, 4))
plot_speed(t, v)
plt.plot(steady_t, steady_v, linestyle="none", marker="s", color="purple",
         label="2–5 min and 10–20 km/h")
plt.title("Combined time-and-speed mask")
plt.legend(fontsize=11)
plt.tight_layout()


<a id="summary"></a>
## Summary and knowledge connection｜总结与知识串联

A Boolean-mask workflow follows a consistent chain:

**data arrays → element-wise comparison → Boolean mask → selection / modification / counting → scientific interpretation**

布尔掩码的完整流程是：从数据数组出发，对每个元素计算条件，得到与数据对齐的布尔数组，再用它完成筛选、修改或计数，最后解释结果的科学含义。

This lesson preserves all major content from the original course: selecting time intervals, selecting by speed, counting matching samples, understanding Boolean arrays, modifying selected values, and distinguishing short-circuit operators from element-wise operators. It also corrects the original typo that listed `gold` instead of `or`, replaces a missing mask image with an explicit table, and makes the duration estimate and operator precedence precise.  
本课保留了原课件的全部核心内容：按时间区间筛选、按速度筛选、统计满足条件的采样点、理解布尔数组、修改选中元素，以及区分短路运算与逐元素运算。同时修正了原课件把 `or` 写成 `gold` 的错误，用明确表格替代缺失的掩码图片，并说明持续时间估算与运算符优先级。

### Reusable checklist｜可复用检查表

1. Does the condition produce the mask I expect?｜条件是否产生了预期掩码？
2. Does the mask have the same shape as the indexed array?｜掩码与被索引数组形状是否一致？
3. Are compound comparisons individually parenthesised?｜组合条件中的每个比较是否分别加了括号？
4. Do I need `&`, `|`, or `~` for arrays rather than `and`, `or`, or `not`?｜数组条件是否应使用 `&`、`|`、`~`？
5. Am I selecting data, modifying it in place, or only counting it?｜我的目标是筛选、原地修改，还是只统计数量？
